# F2 · 02 · Limpieza y transformación

**Qué hace este notebook:**
- toma el dato crudo explorado en F2_01 y lo lleva a un formato limpio y listo para el análisis de la Fase 3.

**Qué no hace:**
- filtrar filas ni decidir tratamientos estadísticos; eso será parte de F3.

Pasos:
1. Registrar el entorno (reproducibilidad).
2. Aplanar las columnas anidadas (`conv_metadata`, `category_tag`).
3. Convertir tipos de dato.
4. Crear variables derivadas para P1 (formato y largo vs. preferencia).
5. Validar que no se perdieron ni duplicaron filas.
6. Guardar el resultado en `data/processed/arena_clean.parquet`.

## 0. Entorno
**Qué hacemos:**
- registrar versiones de Python, librerías, rama y commit.

**Por qué:**
- si algo falla en otro computador, lo primero que se compara son las versiones.

In [1]:
from arena.env import environment_info

info = environment_info()
{k: v for k, v in info.items() if k != "dependencies"}

{'python': '3.12.13',
 'system': 'Darwin 25.6.0',
 'interpreter': '/Users/jose/Developer/magister/programacion_ds/lmarena-sumativa1/.venv/bin/python',
 'virtual_env': True,
 'seed': 42,
 'git': {'branch': 'main', 'commit': '0ccff6a', 'clean': 'False'}}

In [2]:
import pandas as pd

pd.DataFrame(info["dependencies"]).T

,pinned,installed,match
numpy,2.5.4,2.5.4,True
pandas,3.1.0,3.1.0,True
pyarrow,26.0.0,26.0.0,True
matplotlib,3.11.2,3.11.2,True
huggingface_hub,2.4.0,2.4.0,True
pytest,9.2.0,9.2.0,True
nbconvert,7.17.2,7.17.2,True
nbformat,5.11.1,5.11.1,True
ipykernel,7.4.0,7.4.0,True
jupyter_client,8.10.0,8.10.0,True


*Interpretación:* el notebook corre con las dependencias fijadas en `requirements.txt`. Queda registrado el commit y la rama.

## 1. Carga del dato crudo
**Qué hacemos:**
- cargar las 11 columnas del dataset (sin las 3 de conversaciones) con `load_raw`.

**Por qué:**
- es el punto de partida documentado en F2_01.

In [3]:
from arena.profile import load_raw, count_rows

df_raw = load_raw()
n_raw = len(df_raw)
print(f"Filas cargadas: {n_raw:,}")
print(f"Columnas: {list(df_raw.columns)}")
assert n_raw == count_rows()["total"], "El número de filas no coincide con los metadatos"

Filas cargadas: 135,634
Columnas: ['id', 'model_a', 'model_b', 'winner', 'evaluation_session_id', 'evaluation_order', 'conv_metadata', 'category_tag', 'language', 'is_code', 'timestamp']


*Interpretación:* se cargan 135.634 filas y 11 columnas, coincidiendo con lo verificado en F2_01.

---

## 2. Aplanamiento de columnas anidadas
**Qué hacemos:** desempaquetar los diccionarios `conv_metadata` (26 conteos) y `category_tag` (12 valores) en columnas planas.
**Por qué:** pandas trabaja mejor con columnas planas; además, necesitamos acceder a los conteos de encabezados, listas, negritas y tokens para P1.

In [4]:
from arena.preprocessing import flatten

df = flatten(df_raw)
print(f"Columnas antes: {len(df_raw.columns)}")
print(f"Columnas después: {len(df.columns)}")
print(f"Filas conservadas: {len(df):,} (deben ser {n_raw:,})")
assert len(df) == n_raw

Columnas antes: 11
Columnas después: 47
Filas conservadas: 135,634 (deben ser 135,634)


In [5]:
# Nuevas columnas del aplanamiento
new_cols = [c for c in df.columns if c not in df_raw.columns]
print(f"{len(new_cols)} columnas nuevas:")
for c in new_cols:
    print(f"  {c}: {df[c].dtype}")

38 columnas nuevas:
  sum_assistant_a_tokens: int64
  header_a_h1: int64
  header_a_h2: int64
  header_a_h3: int64
  header_a_h4: int64
  header_a_h5: int64
  header_a_h6: int64
  list_a_ordered: int64
  list_a_unordered: int64
  bold_a_asterisk: int64
  bold_a_underscore: int64
  context_a_tokens: int64
  sum_assistant_b_tokens: int64
  header_b_h1: int64
  header_b_h2: int64
  header_b_h3: int64
  header_b_h4: int64
  header_b_h5: int64
  header_b_h6: int64
  list_b_ordered: int64
  list_b_unordered: int64
  bold_b_asterisk: int64
  bold_b_underscore: int64
  context_b_tokens: int64
  sum_user_tokens: int64
  turns: int64
  cat_creative_writing: bool
  cat_creative_writing_score: str
  cat_criteria_complexity: bool
  cat_criteria_creativity: bool
  cat_criteria_domain_knowledge: bool
  cat_criteria_problem_solving: bool
  cat_criteria_real_world: bool
  cat_criteria_specificity: bool
  cat_criteria_technical_accuracy: bool
  cat_if: bool
  cat_if_score: float64
  cat_math: bool


*Interpretación:* `conv_metadata` se expande en 26 columnas y `category_tag` en 12. Los nombres resultantes son legibles: `header_a_h2`, `list_b_unordered`, `bold_a_asterisk`, `cat_criteria_complexity`, etc. No se pierden ni duplican filas.

---

## 3. Conversión de tipos
**Qué hacemos:** asignar los tipos de dato correctos para el análisis.
**Por qué:** `winner` y `language` son categóricas (bajo número de valores distintos), `evaluation_order` cabe en int16, y los booleanos deben ser `bool` de Python.

In [6]:
from arena.preprocessing import convert_types

df = convert_types(df)

# Mostrar los tipos resultantes de las columnas clave
key_cols = ["winner", "language", "is_code", "evaluation_order", "timestamp"]
for c in key_cols:
    print(f"{c}: {df[c].dtype}")

winner: category
language: category
is_code: bool
evaluation_order: int16
timestamp: datetime64[ns]


*Interpretación:* `winner` y `language` pasan a `category` (menor uso de memoria y operaciones más rápidas), `evaluation_order` a `int16` (rango 1–28), `timestamp` ya era `datetime64[ns]`.

---

## 4. Variables derivadas
**Qué hacemos:** crear columnas nuevas que son necesarias para responder P1: ¿el formato y el largo de la respuesta se asocian con ganar el voto?
**Por qué:** las columnas crudas (6 tipos de encabezado, 2 de lista, 2 de negrita por lado) son demasiado granulares para un primer análisis. Las agrupamos en una variable de formato total por lado.

Variables creadas:
| Variable | Descripción |
|---|---|
| `format_a`, `format_b` | Suma de todos los marcadores de formato (headers + listas + negritas) por lado |
| `tokens_log_a`, `tokens_log_b` | log(1 + tokens) para cada lado (la distribución de tokens es muy asimétrica, hallazgo 5 de F2_01) |
| `longer_response` | Cuál lado escribió más tokens: `a`, `b` o `equal` |
| `more_formatted` | Cuál lado usó más formato: `a`, `b` o `equal` |
| `winner_binary` | Simplifica `winner`: model_a→`a`, model_b→`b`, tie/both_bad→`other` |

In [7]:
from arena.preprocessing import derive_variables

df = derive_variables(df)

# Verificar las nuevas columnas
derived = ["format_a", "format_b", "tokens_log_a", "tokens_log_b",
           "longer_response", "more_formatted", "winner_binary"]
for c in derived:
    print(f"{c}: {df[c].dtype}, ejemplo = {df[c].iloc[0]}")

format_a: int64, ejemplo = 112
format_b: int64, ejemplo = 6
tokens_log_a: float32, ejemplo = 7.52564001083374
tokens_log_b: float32, ejemplo = 5.579730033874512
longer_response: str, ejemplo = a
more_formatted: str, ejemplo = a
winner_binary: str, ejemplo = a


In [8]:
# Distribución de format_a
df["format_a"].describe()

count    135634.000000
mean         40.148186
std          65.243573
min           0.000000
25%           5.000000
50%          22.000000
75%          54.000000
max        4233.000000
Name: format_a, dtype: float64

In [9]:
# Distribución de las comparaciones
print(df["longer_response"].value_counts())
print()
print(df["more_formatted"].value_counts())
print()
print(df["winner_binary"].value_counts())

longer_response
b        67641
a        67458
equal      535
Name: count, dtype: int64

more_formatted
a        61877
b        61183
equal    12574
Name: count, dtype: int64

winner_binary
b        49785
a        48563
other    37286
Name: count, dtype: int64


*Interpretación:* las variables derivadas quedan listas. `format_a` y `format_b` resumen el formato en un solo número por lado. La transformación logarítmica normaliza la distribución de tokens. Las comparaciones (`longer_response`, `more_formatted`) simplifican el cruce con `winner_binary` para el análisis de F3.

---

## 5. Validación post-limpieza
**Qué hacemos:**
- verificar que el dato limpio conserva las propiedades del original.

**Por qué:**
- un error silencioso en la transformación puede invalidar todo el análisis posterior.

In [10]:
from arena.validation import validate_dataset

report = validate_dataset(df, expected_rows=n_raw)
report.to_frame()

,status,check,detail
0,OK,required columns,all present
1,OK,no NA in key columns,none
2,OK,unique id,none duplicated
3,OK,winner in allowed set,all valid
4,OK,"evaluation_order integer in [1, 28]",0 out of range
5,OK,timestamp within 2025-04-17..2025-07-24,0 outside
6,OK,sum_assistant_a_tokens >= 0,0 negative
7,OK,sum_assistant_b_tokens >= 0,0 negative
8,OK,sum_user_tokens >= 0,0 negative
9,OK,context_a_tokens >= 0,0 negative


In [11]:
report.assert_ok()
print(f"Fallos críticos: {len(report.failures)}")
print(f"Advertencias: {len(report.warnings)}")

Fallos críticos: 0
Advertencias: 1


In [12]:
# Verificaciones adicionales sobre las variables derivadas
checks = {
    "sin nulos en format_a": df["format_a"].isna().sum() == 0,
    "sin nulos en format_b": df["format_b"].isna().sum() == 0,
    "sin nulos en tokens_log_a": df["tokens_log_a"].isna().sum() == 0,
    "sin nulos en winner_binary": df["winner_binary"].isna().sum() == 0,
    "format >= 0": (df["format_a"] >= 0).all() and (df["format_b"] >= 0).all(),
    "tokens_log >= 0": (df["tokens_log_a"] >= 0).all() and (df["tokens_log_b"] >= 0).all(),
    "winner_binary sin valores inesperados": set(df["winner_binary"].unique()) <= {"a", "b", "other"},
    "longer_response sin valores inesperados": set(df["longer_response"].unique()) <= {"a", "b", "equal"},
}
for name, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"{status} {name}")
assert all(checks.values()), "Alguna verificación falló"

✓ sin nulos en format_a
✓ sin nulos en format_b
✓ sin nulos en tokens_log_a
✓ sin nulos en winner_binary
✓ format >= 0
✓ tokens_log >= 0
✓ winner_binary sin valores inesperados
✓ longer_response sin valores inesperados


*Interpretación:* se conservan las 135.634 filas. Los chequeos del validador pasan sin fallos críticos y con 1 advertencia relacionada a filas en donde se compara el mismo modelo.

## 6. Guardado
**Qué hacemos:**
- guardar el DataFrame limpio en `data/processed/arena_clean.parquet`.

**Por qué:**
- este archivo es el punto de partida de la Fase 3 y de la validación F2_03.

In [13]:
from arena.preprocessing import save_clean

path = save_clean(df)
print(f"Guardado en: {path}")
print(f"Tamaño: {path.stat().st_size / 1e6:.1f} MB")
print(f"Columnas: {len(df.columns)}")
print(f"Filas: {len(df):,}")

Guardado en: /Users/jose/Developer/magister/programacion_ds/lmarena-sumativa1/data/processed/arena_clean.parquet
Tamaño: 15.1 MB
Columnas: 54
Filas: 135,634


In [14]:
# Verificación de lectura: cargar lo guardado y comparar forma
df_check = pd.read_parquet(path)
assert df_check.shape == df.shape, f"Shape mismatch: {df_check.shape} vs {df.shape}"
print(f"Lectura OK: {df_check.shape[0]:,} filas × {df_check.shape[1]} columnas")

Lectura OK: 135,634 filas × 54 columnas


*Interpretación:* el archivo limpio pesa menos que los 7 parquet crudos porque no tiene las columnas de texto ni las estructuras anidadas. Se puede cargar directamente con `pd.read_parquet` para F3.

---

## 7. Resumen de la limpieza

| **Paso** | **Entrada** | **Salida** | **Verificación** |
|---|---|---|---|
| Cargar datos | 7 parquet (data/raw) | 135.634 filas × 11 columnas | Coincide con count_rows |
| Aplanar datos | conv_metadata + category_tag | 135.634 filas × 47 columnas | Sin pérdida de filas |
| Setear tipos | object → category, int32 → int16 | Mismas filas | Tipos correctos |
| Crear variables derivadas | Columnas de formato y tokens | 135.634 filas × 54 columnas | Sin nulos, rangos válidos |
| Validar datos | validate_dataset | 0 fallos críticos | assert_ok pasa |
| Guarda datos limpios | arena_clean.parquet | Mismo shape al leer | Lectura OK |


El dato queda listo para la Fase 3. Las decisiones pendientes (tratamiento de tie/both_bad, filtro por idioma, filas con 0 tokens) se resuelven en F3.